In [2]:
import numpy as np
import pandas as pd

data_dir = '/home/andrew/abm_violence/data/'
land_df = pd.read_csv(f'{data_dir}/county_landmass.csv')
pop_df = pd.read_csv(f'{data_dir}/uscounties.csv')

pop_df['county_state'] = [f'{c}_{s}' for _, c, s in pop_df[['county_ascii', 'state_id']].itertuples()]
land_df['county_state'] = [f'{c}_{s}' for _, c, s in land_df[['county', 'state_abbrev']].itertuples()]
county_df = pop_df.merge(land_df, on='county_state')
county_df = county_df[county_df['sq_mi'] > 0]
county_df['density'] = county_df['population'] / (county_df['land_sq_mi'])

county_df = county_df[["county_x", "state_name", "lat", "lng", "population", "county_state", "density"]]
# county_df.to_csv(f'{data_dir}/county_population.csv')

ffl_df = pd.read_csv(f'{data_dir}/ffl_list.txt', sep='\t')
# ffl_df = ffl_df[ffl_df['LIC_TYPE'] == 1]
ffl_df = ffl_df[['PREMISE_CITY', 'PREMISE_STATE']]
ffl_df['arm_count'] = [1]*len(ffl_df)

city_arm_counts = ffl_df.groupby(['PREMISE_CITY', 'PREMISE_STATE']).count()
city_arm_counts = city_arm_counts.reset_index()
import json
with open(f'{data_dir}/state_acronym.json') as f:
    state_acr = json.load(f)
def title_case(s):
    s = list(s.lower())
    s[0] = s[0].upper()
    for i in range(1, len(s)):
        if(s[i-1] == ' '):
            s[i] = s[i].upper()
    return ''.join(s)

city_arm_counts['state_name'] = [state_acr[s] for s in city_arm_counts['PREMISE_STATE']]
city_arm_counts['city'] = [title_case(s) for s in city_arm_counts['PREMISE_CITY']]
city_arm_counts['city_state'] = [f"{city}, {state}" for city, state in zip(city_arm_counts['city'], city_arm_counts['state_name'])]

city_df = pd.read_csv(f'{data_dir}/uscities.csv')
city_df['city_size'] = city_df['population'] / city_df['density']
city_df['city_state'] = [f"{c}, {s}" for c, s in zip(city_df.city, city_df.state_name)]
city_df = city_df.merge(city_arm_counts, how='left', on='city_state')
city_df['arm_count'] = city_df['arm_count'].fillna(0)
city_df['arm_density'] = city_df['arm_count'] / city_df['city_size']

mj_shootings = pd.read_csv(f"{data_dir}/mother_jones.csv")
mj_shootings = mj_shootings.merge(city_df, how='left', left_on='location', right_on='city_state')

simulation_df = mj_shootings[['location', 'date', 'fatalities', 'injured', 'total_victims', 'population', 'density', 'arm_count', 'arm_density']]
synthetic_df = city_df[['city_state', 'population', 'density', 'arm_count', 'arm_density']].dropna().rename(columns={'city_state': 'location'})
synthetic_df = synthetic_df[synthetic_df['population'] > 100]
synthetic_df = synthetic_df[synthetic_df['density'] > 1]

In [4]:
from gvabm.param_distr import CityParams, EventOutcome
city_data = {c: {'events': []} for c in set(simulation_df['location']).union(synthetic_df['location'])}
for i, row in simulation_df.iterrows():
    c = row.location
    city_data[c]['params'] = CityParams(row.population, row.density, row.arm_count, row.arm_density)
    city_data[c]['events'].append(EventOutcome(row.date, row.fatalities, row.injured, row.total_victims))
for i, row in synthetic_df.iterrows():
    c = row.location
    city_data[c]['params'] = CityParams(row.population, row.density, row.arm_count, row.arm_density)

In [5]:
city_data

{'Princeton, Kansas': {'events': [],
  'params': CityParams(population=208, population_density=206.3, arm_count=1.0, arm_density=0.9918269230769231)},
 'Esmond, North Dakota': {'events': [],
  'params': CityParams(population=110, population_density=93.4, arm_count=0.0, arm_density=0.0)},
 'West Warren, Massachusetts': {'events': [],
  'params': CityParams(population=619, population_density=188.6, arm_count=2.0, arm_density=0.6093699515347334)},
 'Lula, Georgia': {'events': [],
  'params': CityParams(population=2903, population_density=259.7, arm_count=3.0, arm_density=0.26837754047537027)},
 'Harristown, Illinois': {'events': [],
  'params': CityParams(population=1526, population_density=327.9, arm_count=0.0, arm_density=0.0)},
 'Frisco, Pennsylvania': {'events': [],
  'params': CityParams(population=1040, population_density=906.2, arm_count=0.0, arm_density=0.0)},
 'Johnson City, Tennessee': {'events': [],
  'params': CityParams(population=129818, population_density=634.1, arm_count=2

In [ ]:
import pickle
with open(f'{data_dir}/county_data.pkl', 'wb') as f:
    pickle.dump(city_data, f)

In [9]:
known_cs = set((c.lower(), s.lower()) for c, s in zip(city_df['city'], city_df['state_name']))
shootings_cs = []
for l in mj_shootings['location']:
    c, s = tuple(l.split(', '))
    shootings_cs.append((c.lower(), s.lower()))

In [1]:
len(shootings_cs), len(set(shootings_cs)), len(known_cs.intersection(shootings_cs))
print(set(shootings_cs).difference(known_cs.intersection(shootings_cs)))

NameError: name 'shootings_cs' is not defined